# This notebook takes the llm output from nb 005 and outputs an html table

In [1]:
%reload_ext autoreload
%autoreload 2

In [9]:
from rrational.export import export_df_2_html
from rrational.export import export_df_2_html
from pathlib import Path
from rrational.transform import join_uniq, chain_lists, format_flair, c2md, collapsibe, urls2a,url2a,  unique_elements, auto_transform_to_html, long_text_last

from collections import OrderedDict
import pandas as pd

df1 = pd.read_parquet('../outputs/df_links4.parquet')
print(df1.shape)
df = pd.read_parquet('../outputs/df_links5.parquet')
print(df.shape)
# df = df.query('score > 1')
df

(15271, 8)
(13412, 24)


,title_llm,title,score,n_links,n_comments,comments,thread_urls,first_link_utc,last_link_utc,url,...,reccomendations,disrecommendations,why,if_you_liked_x_you_will_like_this,rating_quality,rating_rationality,rating_writing,rating_plot,rating_character,rating_worldbuilding
0,Worth the Candle,Worth the Candle,6082,329,329,"[{'author': 'Noumero', 'author_flair_text': 'S...",[https://reddit.com/r/rational/comments/7dz6kj...,2015-06-05 21:38:35,2024-04-17 19:03:01,[https://archiveofourown.org/works/11478249/ch...,...,Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,Readers of r/rational would likely enjoy Worth...,"[Mother of Learning, HPMOR, Worm, Erfworld, Lu...",9.0,7.0,8.5,8.5,8.0,9.0
1,Mother of Learning,Mother of Learning,2055,187,187,"[{'author': 'TimeLoopedPowerGamer', 'author_fl...",[https://reddit.com/r/rational/comments/cmc4a0...,2014-07-26 08:21:12,2024-05-21 07:04:42,[https://www.fictionpress.com/s/2961893/1/Moth...,...,"Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Readers of r/rational will appreciate the stor...,"[Time Braid, HPMOR, Worm]",9.0,7.5,7.0,9.0,8.5,9.0
2,"Alexander Wales - The Metropolitan Man, Shadow...","Alexander Wales - The Metropolitan Man, Shadow...",1402,22,22,"[{'author': 'alexanderwales', 'author_flair_te...",[https://reddit.com/r/rational/comments/al7z2v...,2015-04-18 18:13:47,2021-04-29 20:00:04,[https://www.patreon.com/alexanderwales],...,Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,Readers of r/rational would likely enjoy Alexa...,"[Brandon Sanderson, Harry Potter, Worm, Mother...",9.0,7.5,9.0,9.0,8.5,8.0
3,A Practical Guide to Evil,A Practical Guide to Evil,1276,139,139,"[{'author': 'Escapement', 'author_flair_text':...",[https://reddit.com/r/rational/comments/byyy3d...,2016-04-05 18:53:21,2024-08-09 10:10:21,"[https://practicalguidetoevil.wordpress.com/, ...",...,"Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,Readers of r/rational will likely enjoy this f...,"[Worm, Shadows of the Limelight]",9.5,7.0,8.0,9.0,9.0,9.0
4,Worm,Worm,1014,97,97,"[{'author': 'ArgentStonecutter', 'author_flair...",[https://reddit.com/r/rational/comments/16rsx5...,2014-01-27 02:09:42,2024-11-22 21:39:11,"[https://parahumans.wordpress.com/, https://pa...",...,Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...",Readers of r/rational might enjoy Worm for its...,"[HPMOR, A Practical Guide to Evil, Other Wildb...",9.0,7.0,8.5,8.5,8.0,9.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13407,r/rational Community Discussion Summary,Much less than or Not at all.,-7,1,1,"[{'author': 'Blusqere', 'author_flair_text': N...",[https://reddit.com/r/rational/comments/kaat79...,2020-12-10 16:35:49,2020-12-10 16:35:49,[https://www.merriam-webster.com/dictionary/no...,...,Some users valued the community and its emphas...,Some users expressed negative associations wit...,Readers interested in discussions about ration...,[],3.0,7.0,0.0,0.0,0.0,0.0
13408,Steelheart (The Reckoners),Steelheart (The Reckoners): 9780385743570: San...,-10,1,1,"[{'author': 'ben_oni', 'author_flair_text': No...",[https://reddit.com/r/rational/comments/7cneg5...,2017-11-13 20:58:01,2017-11-13 20:58:01,[https://www.amazon.com/Steelheart-Reckoners-B...,...,The book is mentioned as a related work to *Wo...,None explicitly stated; the comment thread foc...,Readers of r/rational might appreciate *Steelh...,"[Worm, Mistborn, The Reckoners, The Incredible...",7.0,7.0,7.0,7.0,6.0,8.0
13409,Orthogonality Thesis,Orthogonality thesis,-10,2,2,"[{'author': 'BadGoyWithAGun', 'author_flair_te...",[https://reddit.com/r/rational/comments/3vc0si...,2015-12-04 18:17:07,2016-07-11 16:27:30,[https://wiki.lesswrong.com/wiki/Orthogonality...,...,Users who appreciate discussions of Frie

In [10]:

d = df.copy()#.iloc[:200]

# make title have a link to first url
d['title'] = d.apply(lambda x: f'<a href="{x["url"][0]}">{x["title"]}</a>', axis=1)
d['title_llm'] = d.apply(lambda x: f'<a href="{x["url"][0]}">{x["title_llm"]}</a>', axis=1)

d["url"] = d["url"].apply(lambda x: collapsibe('...', urls2a(x)))
d["score"] = d["score"].round(2)

# TODO unique
d['tags'] = d['tags'].apply(lambda x: ", ".join(unique_elements(x)))
d['reviews_quotes'] = d['reviews_quotes'].apply(lambda x: "- " + "<br> - ".join(unique_elements(x)))
d['if_you_liked_x_you_will_like_this'] = d['if_you_liked_x_you_will_like_this'].apply(lambda x: ", ".join(unique_elements(x)))

# prefix = 'https://reddit.com/r/rational/comments/'
prefix = 'https://reddit.com'
d["comment_urls"] = d['comments'].apply(lambda x: collapsibe("...", urls2a([prefix+y['permalink'] for y in x])))
d["thread_urls"] = d['thread_urls'].apply(lambda x: collapsibe("...", urls2a(x)))
# d['comments'] = d['comments'].progress_apply(lambda x: collapsibe("comments", "<br>".join([c2md(c) for c in x])))


d['first_link_utc'] = d['first_link_utc'].dt.strftime('%Y-%m-%d')
d['last_link_utc'] = d['last_link_utc'].dt.strftime('%Y-%m-%d')

del d['comments']

In [11]:
# QC
d.iloc[0]
print(d.comment_urls.iloc[0][:200])
print()
print(d.thread_urls.iloc[0][:200])
print()
print(d.url.iloc[0][:200])

<details><summary>...</summary>
<ul><li><a href="https://reddit.com/r/rational/comments/6uqc8x/rtwip_worth_the_candle_chapter_21/dluq8vo/">dluq8vo</a></li><li><a href="https://reddit.com/r/rational/co

<details><summary>...</summary>
<ul><li><a href="https://reddit.com/r/rational/comments/7dz6kj/rtwip_worth_the_candle_ch_61_animus/">rtwip_worth_the_candle_ch_61_animus</a></li><li><a href="https://re

<details><summary>...</summary>
<ul><li><a href="https://archiveofourown.org/works/11478249/chapters/25740126">https://archiveofourown.org/works/11478249/chapters/25740126</a></li><li><a href="https:/


In [12]:
from collections import OrderedDict
print(d.columns)
rename_cols = OrderedDict(
    title='Title',
    score='⬆️',
    n_comments='Comments',
    rating_quality='⭐Qual',
    rating_rationality='⭐Rat',
    rating_writing='⭐Writ',
    rating_plot='⭐Plot',
    rating_character='⭐Char',
    rating_worldbuilding='⭐World',
    first_link_utc='First Link',
    last_link_utc='Last Link',
    thread_urls='Threads',
    comment_urls='Comments',
    title_llm='Title (LLM)',
    description='Description',
    tags='Tags',
    why='Why',
    if_you_liked_x_you_will_like_this='Similar',
    reccomendations='Recommendations',
    disrecommendations='Disrecommendations',
    reviews_summary='Reviews Summary',
    reviews_quotes='Reviews',
    url='URLs',
    n_links='Links',
)


dropping = [k for k in d.columns if k not in rename_cols.keys()]
print('Dropping', dropping)

d = d[list(rename_cols.keys())].rename(columns=rename_cols)

Index(['title_llm', 'title', 'score', 'n_links', 'n_comments', 'thread_urls',
       'first_link_utc', 'last_link_utc', 'url', 'description', 'tags',
       'reviews_quotes', 'reviews_summary', 'reccomendations',
       'disrecommendations', 'why', 'if_you_liked_x_you_will_like_this',
       'rating_quality', 'rating_rationality', 'rating_writing', 'rating_plot',
       'rating_character', 'rating_worldbuilding', 'comment_urls'],
      dtype='object')
Dropping []


In [13]:
hidden2 = [
    'title_llm',
    'Title (LLM)',

    'comments',
    'Comments',
    'n_comments',

 'First Link',
 'first_link_utc',

 'Last Link',
 'last_link_utc',

 'Reviews',
 'reviews_quotes',

 'Reviews Summary',
 'reviews_summary',

 '⭐Writ',
 'rating_writing',

 '⭐Plot',
 'rating_plot',

 '⭐Char',
 'rating_character',

 '⭐World',
 'rating_worldbuilding',

 'Recommendations',
 'reccomendations',

 'Disrecommendations',
 'disrecommendations',

 'Links',
 'n_links',

 'Comments',
 'n_comments'
 ]

In [14]:
d = d.sort_values('⬆️', ascending=False)
d

,Title,⬆️,Comments,⭐Qual,⭐Rat,⭐Writ,⭐Plot,⭐Char,⭐World,First Link,...,Description,Tags,Why,Similar,Recommendations,Disrecommendations,Reviews Summary,Reviews,URLs,Links
0,"<a href=""https://archiveofourown.org/works/114...",6082,329,9.0,7.0,8.5,8.5,8.0,9.0,2015-06-05,...,"Worth the Candle is a long, original, complete...","web serial, LitRPG, isekai, portal fantasy, or...",Readers of r/rational would likely enjoy Worth...,"Mother of Learning, HPMOR, Worm, Erfworld, Lum...",Readers frequently praised the story's compell...,Some readers found the sheer length of the wor...,"Overall, Worth the Candle received overwhelmin...","- This is a self-insert litRPG portal fantasy,...",<details><summary>...</summary>\n<ul><li><a hr...,329
1,"<a href=""https://www.fictionpress.com/s/296189...",2055,187,9.0,7.5,7.0,9.0,8.5,9.0,2014-07-26,...,Mother of Learning is a web serial about Zoria...,"web serial, original fiction, fantasy, time lo...",Readers of r/rational will appreciate the stor...,"Time Braid, HPMOR, Worm","Readers praise its consistent internal logic, ...",Some readers find the writing style to be roug...,Generally very positive; praised for clever pr...,- With dread but cautious optimism<br> - Enthu...,<details><summary>...</summary>\n<ul><li><a hr...,187
2,"<a href=""https://www.patreon.com/alexanderwale...",1402,22,9.0,7.5,9.0,9.0,8.5,8.0,2015-04-18,...,"Alexander Wales's works, including *Shadows of...","web serial, rational fiction, fantasy, progres...",Readers of r/rational would likely enjoy Alexa...,"Brandon Sanderson, Harry Potter, Worm, Mother ...",Readers strongly recommend Alexander Wales's w...,Some readers express concerns about the infreq...,"The reviews are overwhelmingly positive, with ...",- Damn. That was like reading some of Sanderso...,<details><summary>...</summary>\n<ul><li><a hr...,22
3,"<a href=""https://practicalguidetoevil.wordpres...",1276,139,9.5,7.0,8.0,9.0,9.0,9.0,2016-04-05,...,A Practical Guide to Evil is a long-running we...,"web serial, fantasy, complete, grimdark, roman...",Readers of r/rational will likely enjoy this f...,"Worm, Shadows of the Limelight","Readers praise its well-developed characters, ...",Some readers find the frequent cliffhangers ir...,"Generally very positive, praising the writing,...",- It's probably one of the better written ones...,<details><summary>...</summary>\n<ul><li><a hr...,139
4,"<a href=""https://parahumans.wordpress.com/"">Wo...",1014,97,9.0,7.0,8.5,8.5,8.0,9.5,2014-01-27,...,"Worm is a long, complete web serial (1.7 milli...","web serial, complete, superhero, supervillain,...",Readers of r/rational might enjoy Worm for its...,"HPMOR, A Practical Guide to Evil, Other Wildbo...",Readers recommend Worm for its compelling char...,"Some find the story too grim, dark, and depres...","Worm receives overwhelmingly positive reviews,...",- I wouldn't really describe Worm as a rationa...,<details><summary>...</summary>\n<ul><li><a hr...,97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13407,"<a href=""https://www.merriam-webster.com/dicti...",-7,1,3.0,7.0,0.0,0.0,0.0,0.0,2020-12-10,...,"This is not a work of fiction, but rather a Re...","reddit discussion, online community, rationali...",Readers interested in discussions about ration...,,Some users valued the community and its emphas...,Some users expressed negative associations wit...,The overall sentiment towards r/rational is mi...,"- ""Much less than or Not at all.""<br> - ""I don...",<details><summary>...</summary>\n<ul><li><a hr...,1
13408,"<a href=""https://www.amazon.com/Steelheart-Rec...",-10,1,7.0,7.0,7.0,7.0,6.0,8.0,2017-11-13,...,"Steelheart, the first book in Brandon Sanderso...","novel, complete, superhero, superpower, decons...",Readers of r/rational might appreciate *Steelh...,"Worm, Mistborn, The Reckoners, The Incredibles...",The book is mentioned as a related work to *Wo...,None explicitly stated; the comment thread foc...,"Steelheart is mentioned in comparison 

In [15]:


# d = d.copy().drop(columns=['comments'])
# df_rr2 = long_text_last(df_rr2)

# df_rr2 = auto_transform_to_html(df_rr2)

# export_df_2_html(df=df_rr2, output=Path('../outputs/df_links5.html'))

export_df_2_html(
    template="../index.jinja2.html",
    df=d,
    columns=hidden2,
)